# ForestGuard

## 02 — Limpieza, transformación e integración

Homologación, limpieza e integración de los registros 2022–2025.

## 1. Carga de los datos originales

In [ ]:
from pathlib import Path
from google.colab import files
import pandas as pd
import numpy as np
import time

ROOT = Path("/content")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

archivo_historico = RAW / "incendios_2022_2024_raw.csv"
archivo_2025 = RAW / "incendios_2025_raw.csv"

if not archivo_historico.exists() or not archivo_2025.exists():
    print("Sube los dos archivos generados por el módulo 01:")
    print("- incendios_2022_2024_raw.csv")
    print("- incendios_2025_raw.csv")

    subidos = files.upload()

    for nombre, contenido in subidos.items():
        with open(RAW / nombre, "wb") as f:
            f.write(contenido)

if not archivo_historico.exists() or not archivo_2025.exists():
    raise FileNotFoundError(
        "Faltan los archivos raw. Ejecuta el módulo 01 y sube los dos CSV."
    )

df_hist = pd.read_csv(archivo_historico)
df_2025 = pd.read_csv(archivo_2025)

print("Histórico cargado:", df_hist.shape)
print("2025 cargado:", df_2025.shape)

## 2. Homologación de variables

Las dos fuentes representan el mismo tipo de fenómeno, pero utilizan nombres distintos para varias columnas. Se crea una estructura común para poder concatenarlas correctamente.

Las variables comunes utilizadas son:

- `año`: año del evento.
- `mes` y `dia`: momento del registro.
- `provincia`, `canton` y `parroquia`: ubicación administrativa.
- `latitud` y `longitud`: ubicación geográfica.
- `region`: región natural derivada de la provincia.
- `causa`: causa registrada.
- `hadecobert`: hectáreas de cobertura vegetal afectada.
- `nivelinf`: nivel de afectación.
- `personas_afectadas`, `familias_afectadas`, `animales_afectados` y `animales_muertos`: variables adicionales de impacto.
- `fuente_dataset`: permite identificar el origen de cada registro.

In [ ]:
def primera_columna_existente(df, candidatas):
    return next((c for c in candidatas if c in df.columns), None)

# Resolver nombres reales de la fuente histórica
columnas_hist = {
    "año": primera_columna_existente(df_hist, ["año", "anio"]),
    "mes": primera_columna_existente(df_hist, ["mes"]),
    "dia": primera_columna_existente(df_hist, ["dia_evento", "dia"]),
    "provincia": primera_columna_existente(df_hist, ["provincia"]),
    "canton": primera_columna_existente(df_hist, ["canton", "cantón"]),
    "parroquia": primera_columna_existente(df_hist, ["parroquias", "parroquia"]),
    "latitud": primera_columna_existente(df_hist, ["latitud", "latitude"]),
    "longitud": primera_columna_existente(df_hist, ["longitud", "longitude"]),
    "causa": primera_columna_existente(df_hist, ["causa"]),
    "hadecobert": primera_columna_existente(
        df_hist,
        [
            "ha__de_cob",
            "ha_de_cob",
            "hadecobert",
            "ha_de_cobertura_vegetal_quemada",
            "ha__de_cobertura_vegetal_quemad",
        ]
    ),
    "nivelinf": primera_columna_existente(
        df_hist,
        ["nivel_del_", "nivel_del_inf", "nivelinf"]
    ),
    "personas_afectadas": primera_columna_existente(
        df_hist,
        ["personas_a", "personas_afectadas_directamente", "personasaf"]
    ),
    "familias_afectadas": primera_columna_existente(
        df_hist,
        ["familias_a", "familias_afectadas", "familiasaf"]
    ),
    "animales_afectados": primera_columna_existente(
        df_hist,
        ["animales_a", "animales_afectados", "animalesaf"]
    ),
    "animales_muertos": primera_columna_existente(
        df_hist,
        ["animales_m", "animales_muertos", "animalesmu"]
    ),
}

print("Correspondencia de columnas históricas:")
for destino, origen in columnas_hist.items():
    print(f"{destino}: {origen}")

# Crear dataframe histórico homologado
df_hist_h = pd.DataFrame(index=df_hist.index)

for destino, origen in columnas_hist.items():
    if origen is not None:
        df_hist_h[destino] = df_hist[origen]
    else:
        df_hist_h[destino] = np.nan

# Homologar 2025
mapa_2025 = {
    "dia": "dia",
    "provincia": "provincia",
    "canton": "canton",
    "parroquia": "parroquia",
    "latitud": "latitud",
    "longitud": "longitud",
    "mes": "mes",
    "causa": "causa",
    "hadecobert": "hadecobert",
    "nivelinf": "nivelinf",
    "personasaf": "personas_afectadas",
    "familiasaf": "familias_afectadas",
    "animalesaf": "animales_afectados",
    "animalesmu": "animales_muertos",
}

df_2025_h = df_2025.rename(columns=mapa_2025).copy()
df_2025_h["año"] = 2025

columnas_comunes = [
    "año",
    "mes",
    "dia",
    "provincia",
    "canton",
    "parroquia",
    "latitud",
    "longitud",
    "causa",
    "hadecobert",
    "nivelinf",
    "personas_afectadas",
    "familias_afectadas",
    "animales_afectados",
    "animales_muertos",
]

for columna in columnas_comunes:
    if columna not in df_2025_h.columns:
        df_2025_h[columna] = np.nan

df_hist_h = df_hist_h[columnas_comunes].copy()
df_2025_h = df_2025_h[columnas_comunes].copy()

df_hist_h["fuente_dataset"] = "Histórico 2022-2024"
df_2025_h["fuente_dataset"] = "Incendios 2025"

print("Histórico homologado:", df_hist_h.shape)
print("2025 homologado:", df_2025_h.shape)

## 3. Estandarización de meses

La fuente histórica puede almacenar el mes como nombre o texto, mientras que la fuente 2025 puede utilizar valores numéricos. Se convierte todo a números del 1 al 12.

In [ ]:
MESES = {
    "ENERO": 1,
    "FEBRERO": 2,
    "MARZO": 3,
    "ABRIL": 4,
    "MAYO": 5,
    "JUNIO": 6,
    "JULIO": 7,
    "AGOSTO": 8,
    "SEPTIEMBRE": 9,
    "SETIEMBRE": 9,
    "OCTUBRE": 10,
    "NOVIEMBRE": 11,
    "DICIEMBRE": 12,
}

def normalizar_mes(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().upper()

    if texto in MESES:
        return MESES[texto]

    numero = pd.to_numeric(texto, errors="coerce")

    if pd.notna(numero) and 1 <= numero <= 12:
        return int(numero)

    return np.nan

df_hist_h["mes"] = df_hist_h["mes"].apply(normalizar_mes)
df_2025_h["mes"] = df_2025_h["mes"].apply(normalizar_mes)

## 4. Integración vertical de los datasets

Después de homologar las columnas, los registros de 2022–2024 y 2025 se concatenan verticalmente. Cada fila continúa representando un incendio forestal registrado.

In [ ]:
df_integrado = pd.concat(
    [df_hist_h, df_2025_h],
    ignore_index=True
)

df_original = df_integrado.copy()

print("Dataset integrado:", df_integrado.shape)

display(
    df_integrado["año"]
    .value_counts()
    .sort_index()
    .to_frame("Cantidad de registros")
)

### Validación de la integración 2022–2025

Antes de continuar con el análisis exploratorio y Machine Learning se comprueba que el dataset integrado contiene registros de los cuatro años esperados.

Si falta alguno de los años, el proceso se detiene para evitar entrenar modelos con una integración incompleta.

In [ ]:
conteo_integrado_anio = (
    df_integrado["año"]
    .value_counts()
    .sort_index()
)

display(
    conteo_integrado_anio.to_frame(
        "Cantidad de registros"
    )
)

anios_integrados = set(
    pd.to_numeric(df_integrado["año"], errors="coerce")
    .dropna()
    .astype(int)
    .unique()
)

anios_esperados = {2022, 2023, 2024, 2025}
faltantes = anios_esperados - anios_integrados

if faltantes:
    raise ValueError(
        f"La integración está incompleta. Faltan los años: {sorted(faltantes)}"
    )

print("Integración correcta: se encontraron datos de 2022, 2023, 2024 y 2025.")
print("Total de registros integrados:", len(df_integrado))

## 5. Creación de la variable región

Para disponer de una variable geográfica comparable en todos los años, la región natural se deriva a partir de la provincia.

In [ ]:
REGIONES = {
    # Costa
    "ESMERALDAS": "Costa",
    "MANABI": "Costa",
    "MANABÍ": "Costa",
    "SANTO DOMINGO DE LOS TSACHILAS": "Costa",
    "SANTO DOMINGO DE LOS TSÁCHILAS": "Costa",
    "LOS RIOS": "Costa",
    "LOS RÍOS": "Costa",
    "GUAYAS": "Costa",
    "SANTA ELENA": "Costa",
    "EL ORO": "Costa",

    # Sierra
    "CARCHI": "Sierra",
    "IMBABURA": "Sierra",
    "PICHINCHA": "Sierra",
    "COTOPAXI": "Sierra",
    "TUNGURAHUA": "Sierra",
    "CHIMBORAZO": "Sierra",
    "BOLIVAR": "Sierra",
    "BOLÍVAR": "Sierra",
    "CAÑAR": "Sierra",
    "AZUAY": "Sierra",
    "LOJA": "Sierra",

    # Amazonía
    "SUCUMBIOS": "Amazonía",
    "SUCUMBÍOS": "Amazonía",
    "NAPO": "Amazonía",
    "ORELLANA": "Amazonía",
    "PASTAZA": "Amazonía",
    "MORONA SANTIAGO": "Amazonía",
    "ZAMORA CHINCHIPE": "Amazonía",

    # Insular
    "GALAPAGOS": "Insular",
    "GALÁPAGOS": "Insular",
}

provincia_normalizada = (
    df_integrado["provincia"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_integrado["region"] = provincia_normalizada.map(REGIONES)

display(
    df_integrado["region"]
    .value_counts(dropna=False)
    .to_frame("Cantidad")
)

## 6. Limpieza de textos, vacíos y duplicados

In [ ]:
columnas_texto = df_integrado.select_dtypes(include="object").columns

for columna in columnas_texto:
    df_integrado[columna] = (
        df_integrado[columna]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

duplicados_antes = df_integrado.duplicated().sum()

df_limpio = (
    df_integrado
    .drop_duplicates()
    .copy()
)

print("Duplicados antes:", duplicados_antes)
print("Duplicados después:", df_limpio.duplicated().sum())

## 7. Conversión de tipos de datos

In [ ]:
columnas_numericas = [
    "año",
    "mes",
    "dia",
    "latitud",
    "longitud",
    "hadecobert",
    "personas_afectadas",
    "familias_afectadas",
    "animales_afectados",
    "animales_muertos",
]

for columna in columnas_numericas:
    df_limpio[columna] = pd.to_numeric(
        df_limpio[columna],
        errors="coerce"
    )

df_limpio = df_limpio[
    df_limpio["año"].between(2022, 2025, inclusive="both")
].copy()

print(df_limpio.dtypes)

## 8. Revisión de valores faltantes

In [ ]:
faltantes = pd.DataFrame({
    "Valores faltantes": df_limpio.isnull().sum(),
    "Porcentaje": (df_limpio.isnull().mean() * 100).round(2),
}).sort_values("Valores faltantes", ascending=False)

display(faltantes)

## 9. Revisión de valores atípicos en hectáreas afectadas

Los valores atípicos se identifican mediante el rango intercuartílico, pero no se eliminan automáticamente porque pueden representar incendios reales de gran magnitud.

In [ ]:
area = df_limpio["hadecobert"].dropna()

if not area.empty:
    q1 = area.quantile(0.25)
    q3 = area.quantile(0.75)
    iqr = q3 - q1
    limite_superior = q3 + 1.5 * iqr

    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Límite superior:", limite_superior)
    print(
        "Registros sobre el límite:",
        (area > limite_superior).sum()
    )
else:
    print("No existen valores disponibles en hadecobert.")

## 10. Comparación antes y después

Esta tabla resume la calidad del dataset integrado antes y después de aplicar las transformaciones principales.

In [ ]:
comparacion = pd.DataFrame({
    "Indicador": [
        "Registros",
        "Columnas",
        "Duplicados",
        "Valores faltantes reconocidos",
    ],
    "Antes": [
        df_original.shape[0],
        df_original.shape[1],
        df_original.duplicated().sum(),
        int(df_original.isnull().sum().sum()),
    ],
    "Después": [
        df_limpio.shape[0],
        df_limpio.shape[1],
        df_limpio.duplicated().sum(),
        int(df_limpio.isnull().sum().sum()),
    ],
})

display(comparacion)

## 11. Comparación de alternativas de procesamiento

Para documentar la comparación solicitada en el proyecto se realiza el mismo conteo por provincia mediante un ciclo tradicional y mediante una operación vectorizada de pandas.

En un dataset de este tamaño la diferencia puede ser pequeña; el objetivo es mostrar dos formas de procesamiento y comprobar que producen resultados equivalentes.

In [ ]:
inicio = time.perf_counter()

conteo_manual = {}

for provincia in df_limpio["provincia"].dropna():
    conteo_manual[provincia] = conteo_manual.get(provincia, 0) + 1

tiempo_manual = time.perf_counter() - inicio

inicio = time.perf_counter()

conteo_pandas = df_limpio["provincia"].value_counts()

tiempo_pandas = time.perf_counter() - inicio

comparacion_procesamiento = pd.DataFrame({
    "Alternativa": [
        "Ciclo tradicional",
        "Operación de pandas",
    ],
    "Tiempo de ejecución (segundos)": [
        tiempo_manual,
        tiempo_pandas,
    ],
})

display(comparacion_procesamiento)

## 12. Guardado del dataset integrado y limpio

In [ ]:
archivo_processed = (
    PROCESSED /
    "incendios_forestales_2022_2025_limpio.csv"
)

df_limpio.to_csv(
    archivo_processed,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset procesado guardado en:", archivo_processed)
print("Dimensiones finales:", df_limpio.shape)

## 13. Variables disponibles para etapas posteriores

La nueva versión incorpora el año del evento y variables adicionales de impacto humano y animal. Esto permite ampliar los análisis posteriores y preparar una versión más rica del dataset para Machine Learning.

In [ ]:
variables_candidatas = [
    "año",
    "mes",
    "dia",
    "provincia",
    "canton",
    "parroquia",
    "region",
    "latitud",
    "longitud",
    "causa",
    "personas_afectadas",
    "familias_afectadas",
    "animales_afectados",
    "animales_muertos",
]

variables_candidatas = [
    c for c in variables_candidatas
    if c in df_limpio.columns
]

print("Variables candidatas:")
print(variables_candidatas)

print("\nPosibles objetivos:")
print("- hadecobert: regresión")
print("- nivelinf: clasificación")

## 14. Conclusión

La preparación amplía ForestGuard de una sola fuente correspondiente a 2025 a un conjunto integrado de registros entre 2022 y 2025.

También se homologaron variables adicionales de impacto, ubicación y tiempo. El dataset resultante queda estructurado para el análisis exploratorio y para futuras actualizaciones de los modelos de Machine Learning.

## Descarga del dataset procesado

Descarga el CSV final para utilizarlo en los módulos 03, 04 y 05.

In [ ]:
from google.colab import files

files.download(str(archivo_processed))